### Imports


In [109]:
# !pip install google-genai scipy boto3 -q

In [110]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_fscore_support,
)
import re
import math
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import signal
import subprocess
import tempfile
import textwrap

from scipy.stats import binomtest, norm

import multiprocessing
import traceback
import hashlib

from typing import List, Dict, Any, Tuple, Callable, Optional, Union

### Parameter Cofiguration


In [111]:
# Experiment parameters
EXPERIMENT_NUMBER = "expI"
EXP_VERSION = "v1"
GENERATION_TYPE = (
    "during"                # 'during' or 'after' or 'only-gen' or 'without-correctness'
)
DATASET = "humaneval"

In [112]:
MODEL_NAME = "claude"
MODEL_PATH = "gemini-2.5-flash"

# Project root directory (absolute path to avoid confusion with relative paths)
BASE_DIR = "/home/fahad/Documents/RESEARCH_CODEBASE/promptmark"  # or '.'

# Derived paths
if DATASET == "humaneval":
    DATASET_PATH = f"{BASE_DIR}/datasets/human_eval_164.jsonl"
else:
    DATASET_PATH = (
        f"{BASE_DIR}/datasets/sanitized-mbpp-sample-100.json"
    )

# Watermark parameters
Z_THRESHOLD = 1.96                  # corresponds to a p-value for one tailed test is approximately 0.025
P_THRESHOLD = norm.sf(Z_THRESHOLD)

SEED_KEY = "exp2025"
SMALL_SAMPLE_THRESHOLD = 30
ITER_CAP = 5

In [113]:
COMMENT_ENABLED = True          # If True: include comments in generated code; If False: generate code without comments
CHECK_CORRECTNESS = True        # If True: evaluate correctness (tests passed); If False: skip correctness check
APPLY_WATERMARKING = True       # If True: apply watermarking constraints to prompt; If False: generic code generation
ITERATIVE_MODE = True           # If True: generate with feedback loops (iterative); If False: generate once (one-shot)

In [ ]:
df = pd.read_json(DATASET_PATH, lines=True)

# sample up to 2 records reproducibly using SEED_KEY
n = min(2, len(df))
df = df.sample(n=n, random_state=38).reset_index(drop=True)

print("Columns: ", df.columns.to_list())
print("Size: ", len(df))
print("ID: ", df["task_id"].tolist())

Columns:  ['task_id', 'prompt', 'code', 'test', 'entry_point']
Size:  2
ID:  [39, 85]


In [115]:
SAMPLE_SIZE = df.shape[0]

RESULTS_CSV = f"{BASE_DIR}/results/raw/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}.csv"
OUTPUT_DIR = f"{BASE_DIR}/output/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}"

### Helper Functions 

In [116]:
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import keyword

COMMON_STD_METHODS = {
    "self",
    "re",
    "cls",
    "append",
    "join",
    "dummy_function",
    "find",
    "kwargs",
    "args",
    "range",
    "print",
    "len",
    "dict",
    "list",
    "str",
    "int",
    "float",
    "set",
    "tuple",
    "os",
    "np",
    "subprocess",
    "now",
    "today",
    "timedelta",
    "strptime",
    "date",
    "time",
    "datetime",
    "logging",
    "log",
    "info",
    "debug",
    "error",
    "warning",
    "exception",
    "lower",
    "upper",
    "strip",
    "split",
    "replace",
    "endswith",
    "startswith",
    "append",
    "extend",
    "insert",
    "remove",
    "pop",
    "sort",
    "clear",
    "keys",
    "values",
    "items",
    "get",
    "update",
    "copy",
    "format",
    "join",
    "count",
    "index",
}

BUILTIN_NAMES = set(dir(builtins)).union(COMMON_STD_METHODS)


#? CODE ANALYSIS AND TOKEN EXTRACTION
class CodeNavigator(ast.NodeVisitor):
    def __init__(self):
  
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()
        self.docstrings = []

    def visit_FunctionDef(self, node):
        name = node.name

        # classify
        if name.startswith("__") and name.endswith("__"):
            pass
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        # add arguments
        for arg in node.args.args:
            if arg.arg not in BUILTIN_NAMES:
                self.non_public_vars.add(arg.arg)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "function_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.non_public_vars.add(node.name)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "class_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id not in BUILTIN_NAMES:
                if target.id.isupper():
                    self.public_vars.add(target.id)
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        # Detect method calls like self.get_possible_moves
        if isinstance(node.value, ast.Name) and node.value.id == "self":
            if node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
                # treat as a public method
                self.public_funcs.add(node.attr)
        elif node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
            # treat other attributes normally
            self.non_public_vars.add(node.attr)
        self.generic_visit(node)

    def visit_Name(self, node):
        if node.id not in BUILTIN_NAMES:
            self.non_public_vars.add(node.id)
        self.generic_visit(node)

    def visit_Module(self, node):
        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "module_docstring",
                    "name": "__main__",
                    "line_number": getattr(node, "lineno", 1),
                    "content": doc,
                }
            )
        self.generic_visit(node)


def get_tokens(code):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return set()

    visitor = CodeNavigator()
    visitor.visit(tree)

    all_tokens = (
        visitor.public_classes
        # | visitor.public_funcs
        | visitor.non_public_funcs
        | visitor.non_public_vars
        | visitor.non_public_classes
    )

    # 🚫 Remove known stdlib or logging-related names
    cleaned_tokens = {
        t for t in all_tokens if t not in COMMON_STD_METHODS and t not in BUILTIN_NAMES
    }

    return cleaned_tokens


def load_generated_code(file_path):
    if not os.path.exists(file_path):
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    return content.strip() if content.strip() else None


def load_character_frequency_list(green_letters, base_dir):

    # humaneval freqs
    he_freq_path = os.path.join(base_dir, "results/dataset/humaneval_letter_freqs.json")
    mbpp_freq_path = os.path.join(base_dir, "results/dataset/mbpp_letter_freqs.json")

    letter_freqs = defaultdict(int)
    total_identifiers = 0

    if os.path.exists(he_freq_path):
        with open(he_freq_path, "r") as f:
            df_he = json.load(f)
            df_he = df_he["letter_freqs"]
            # print('Values: ', df_he)

            for letter in green_letters:
                letter_freqs[letter] += df_he.get(letter, 0)
            total_identifiers += sum(df_he.values())

    if os.path.exists(mbpp_freq_path):
        with open(mbpp_freq_path, "r") as f:
            df_mbpp = json.load(f)
            df_mbpp = df_mbpp["letter_freqs"]

            for letter in green_letters:
                letter_freqs[letter] += df_mbpp.get(letter, 0)
            total_identifiers += sum(df_mbpp.values())

    return letter_freqs, total_identifiers


#? EXTRACT STARTING LETTERS FROM COMMENTS
def extract_comments_from_source(source_code: str) -> List[Dict[str, Any]]:
    comments = []

    # create a deepcopy
    import copy

    source_code = copy.deepcopy(source_code)

    # Split into lines to process each line individually
    lines = source_code.split("\n")

    for line_number, line in enumerate(lines, 1):
        # Find all # symbols and extract comments after them
        hash_index = line.find("#")
        if hash_index != -1:
            # Extract everything after the # symbol
            comment_content = line[hash_index + 1 :].strip()
            if comment_content:  # Skip empty comments
                # Determine if it's an inline comment or full-line comment
                code_before_hash = line[:hash_index].strip()
                comment_type = (
                    "inline_comment" if code_before_hash else "full_line_comment"
                )

                comments.append(
                    {
                        "line_number": line_number,
                        "content": comment_content,
                        "type": comment_type,
                        "code_context": (
                            code_before_hash[:50] + "..."
                            if len(code_before_hash) > 50
                            else code_before_hash
                        ),
                    }
                )
    # Also extract docstrings using AST visitor
    tree = ast.parse(source_code)
    visitor = CodeNavigator()
    visitor.visit(tree)

    docstrings = visitor.docstrings

    comments.extend(docstrings)

    return comments


def get_comment_starting_letters(source_code: str, gamma) -> tuple:

    try:
        comments = extract_comments_from_source(source_code)

        # print(f"EXTRACTED COMMENTS:\n {[comment['content'] for comment in comments]}\n")

        starting_letters = []
        relevant_words = set()

        for comment in comments:
            # Split comment content by newlines to handle multi-line comments
            comment_lines = comment["content"].split("\n")

            for line in comment_lines:
                line = line.strip()
                if not line:
                    continue

                # Extract the first word from this line
                words = re.findall(r"\b[a-zA-Z]+\b", line)
                if words:
                    first_word = words[0].lower()
                    relevant_words.add(first_word)

                    # Get starting letter of the first word
                    if first_word:
                        first_char = first_word[0].lower()
                        if first_char.isalpha():
                            starting_letters.append(first_char)

        # print(f"Relevant words: {len(relevant_words)}")

        # Use global green_letters and gamma
        green_letters, red_letters, _ = get_red_green_sets(
            secret_key=SEED_KEY, base_dir=BASE_DIR
        )
        green_count = sum(1 for letter in starting_letters if letter in green_letters)
        token_count = len(starting_letters)

        if token_count > 0:
            p_hat = green_count / token_count
            z_score = (p_hat - gamma) / (
                (gamma * (1 - gamma)) ** 0.5 / token_count**0.5
            )
            p_value = norm.sf(z_score)  # one-tailed test
        else:
            z_score = 0.0
            p_value = 1.0

        return starting_letters, relevant_words, green_count, z_score, p_value

    except Exception as e:
        print(f"❌ Error extracting comment letters: {type(e).__name__}: {e}")
        import traceback

        traceback.print_exc()
        return [], set(), 0, 0.0, 1.0


#? EXTRACT FUNCTION NAMES AND FIX METHOD NAME
def extract_function_names_from_code(code: str):
    """Extract all function names defined in the user code."""
    tree = ast.parse(code)
    return [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]


def extract_function_name_from_tests(test_list):
    """Extract the function name used in assert statements."""
    for test in test_list:
        try:
            tree = ast.parse(test)
            for node in ast.walk(tree):
                # Detect function call inside assert or math.isclose( func(...) )
                if isinstance(node, ast.Call):
                    # Handle nested calls like math.isclose(func(...))
                    for arg in node.args:
                        if isinstance(arg, ast.Call) and isinstance(arg.func, ast.Name):
                            return arg.func.id
                    if isinstance(node.func, ast.Name):
                        return node.func.id
        except SyntaxError:
            continue
    return None


def replace_function_name(code, old_name, new_name):
    """Safely rename the function in the code using AST."""

    class RenameTransformer(ast.NodeTransformer):
        def visit_FunctionDef(self, node):
            if node.name == old_name:
                node.name = new_name
            return node

    tree = ast.parse(code)
    tree = RenameTransformer().visit(tree)
    ast.fix_missing_locations(tree)
    return ast.unparse(tree)


def update_method_name(user_code, test_list):
    """If needed, rename user's function to match test case function name."""
    code_func_names = extract_function_names_from_code(user_code)
    test_func_name = extract_function_name_from_tests(test_list)

    if not test_func_name:
        print("⚠️ No function found in test cases.")
        return user_code

    # Case 1: If test function name already exists in code, no change needed
    if test_func_name in code_func_names:
        return user_code

    # Case 2: Otherwise, rename the first user function to match test case
    if code_func_names:
        old_name = code_func_names[0]
        updated_code = replace_function_name(user_code, old_name, test_func_name)
        print(f"🔄 Renamed '{old_name}' → '{test_func_name}'")
        return updated_code

    print("⚠️ No function found in user code.")
    return user_code


#? EXTRACT STARTING LETTERS FROM IDENTIFIERS
def get_starting_character(token):
    if not token:
        return None
    if token[0] == "_":
        if len(token) > 1 and token[1].isalpha():
            return token[1].lower()
        else:
            return None
    elif token[0].isalpha():
        return token[0].lower()
    else:
        return None


def get_starting_letters(code, gamma):
    code = textwrap.dedent(code)

    try:
        tree = ast.parse(code)
    except SyntaxError:
        return {
            "id_starting_letters": [],
            "id_unique_tokens": set(),
            "id_green_count": 0,
            "id_token_count": 0,
            "comment_starting_letters": [],
            "comment_unique_tokens": set(),
            "comment_green_count": 0,
            "comment_token_count": 0,
            "total_starting_letters": [],
            "total_green_count": 0,
            "total_token_count": 0,
            "overlap_count": 0,
            "p_exact": 1.0,
            "score": 0.0,
            "z_score": 0.0,
        }

    visitor = CodeNavigator()
    visitor.visit(tree)

    #? PHASE 1: IDENTIFIER TOKENS
    non_public_tokens = (
        visitor.non_public_classes
        | visitor.non_public_funcs
        | visitor.non_public_vars
    )

    relevant_tokens = [
        token for token in non_public_tokens if token not in {"self", "cls"}
    ]

    id_starting_letters = [
        letter
        for token in relevant_tokens
        if (letter := get_starting_character(token)) is not None
    ]

    id_green_count = sum(
        1 for letter in id_starting_letters if letter in green_letters
    )
    id_token_count = len(id_starting_letters)
    id_unique_tokens = set(relevant_tokens)

    #? PHASE 2: COMMENT TOKENS  
    comment_data = {}
    comment_starting_letters = []
    comment_unique_tokens = set()
    comment_green_count = 0

    if COMMENT_ENABLED:
        cmnt_starting_letters, cmn_relevant_words, cmnt_green_count, _, _ = (
            get_comment_starting_letters(code, gamma)
        )
        comment_starting_letters = cmnt_starting_letters
        comment_unique_tokens = set(cmn_relevant_words)
        comment_green_count = cmnt_green_count

    comment_token_count = len(comment_starting_letters)

    #? PHASE 3: MERGE WITH DEDUPLICATION
   
    # Count overlap: tokens that appear in both identifiers and comments
    overlap_tokens = id_unique_tokens & comment_unique_tokens
    total_starting_letters = id_starting_letters + comment_starting_letters
    total_green_count = id_green_count + comment_green_count
    total_token_count = id_token_count + comment_token_count

    # Number of distinct token names that appear in both phases
    overlap_count = len(overlap_tokens)

    #? CALCULATE p_exact, z_score, AND score =====
    if total_token_count == 0:
        p_exact = 1.0
        z_score = 0.0
        score = 0.0
    else:
        detection_stats = get_detection_score(
            total_token_count, total_green_count, gamma
        )
        p_exact = detection_stats["p_exact"]
        score = detection_stats["score"]
        z_score = calculate_z_score(total_token_count, total_green_count, gamma)

    return {
        # Identifier metrics
        "id_starting_letters": id_starting_letters,
        "id_unique_tokens": id_unique_tokens,
        "id_green_count": id_green_count,
        "id_token_count": id_token_count,
        # Comment metrics
        "comment_starting_letters": comment_starting_letters,
        "comment_unique_tokens": comment_unique_tokens,
        "comment_green_count": comment_green_count,
        "comment_token_count": comment_token_count,
        # Combined metrics
        "total_starting_letters": total_starting_letters,
        "total_green_count": total_green_count,
        "total_token_count": total_token_count,
        "overlap_count": overlap_count,
        # Detection stats
        "p_exact": p_exact,
        "z_score": z_score,
        "score": score,
    }


def extract_identifier_starting_letters(code: str) -> Tuple[List[str], set]:
    """
    Extract user-defined identifiers from source code and return their starting letters and identifiers.
    """
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return [], set()
    
    visitor = CodeNavigator()
    visitor.visit(tree)
    
    # Collect all valid user-defined identifiers
    all_identifiers = (
        visitor.public_classes
        | visitor.non_public_classes
        | visitor.non_public_funcs
        | visitor.non_public_vars
        | visitor.public_funcs
        | visitor.public_vars
    )
    
    # Filter out known stdlib/logging names
    valid_identifiers = {
        ident for ident in all_identifiers 
        if ident not in COMMON_STD_METHODS and ident not in BUILTIN_NAMES
    }
    
    # Extract first alphabetic letter from each identifier
    starting_letters = []
    for ident in valid_identifiers:
        # Skip identifiers starting with underscore/digits
        for char in ident:
            if char.isalpha():
                starting_letters.append(char.lower())
                break
    
    return starting_letters, valid_identifiers


#? TEST EXECUTION FUNCTIONS 
def run_code_with_tests(code, test_imports, tests, return_dict):
    """Execute code with test assertions and track results."""
    try:
        # Shared environment for both code and tests
        env = {}

        # Run any imports from test_imports
        for imp in test_imports:
            exec(imp, env, env)

        # Execute user code
        exec(code, env, env)

        passed, failed = 0, 0
        return_dict["error"] = ""  # Initialize error as empty string

        # Run all test assertions
        for t in tests:
            try:
                exec(t, env, env)
                passed += 1
            except AssertionError:
                failed += 1
                return_dict["error"] += f"Assertion Error in: {t}\n"
            except Exception as e:
                failed += 1
                return_dict["error"] += f"Exception Error in: {t} → {e}\n"

        return_dict["result"] = (passed, failed)

    except Exception as e:
        tb = traceback.format_exc()
        return_dict["error"] = f"Runtime Error in user code:\n{tb}"

    return return_dict


def safe_exec_with_tests(code, test_imports, tests, timeout=2):
    """
    Safely execute code with tests using multiprocessing for timeout handling.
    Works for both HumanEval and MBPP datasets.
    """
    manager = multiprocessing.Manager()
    return_dict = manager.dict()
    p = multiprocessing.Process(
        target=run_code_with_tests, args=(code, test_imports, tests, return_dict)
    )

    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        print("⏰ Timeout: possible infinite loop or hanging code.")
        return_dict["result"] = (0, len(tests))
        return_dict["error"] = "Timeout: possible infinite loop or hanging code"
        return return_dict

    if "error" in return_dict:
        return return_dict

    return return_dict


def extract_assertions_from_check_function(test_code_str, entry_point_name=None):
    """
    Extract assertion statements from a HumanEval check() function.
    Replace 'candidate' calls with the actual function name if entry_point_name is provided.
    
    HumanEval format: def check(candidate)...assert candidate(...)...; 
    Returns: List of assertion strings that can be directly executed.
    """
    assertions = []
    
    try:
        tree = ast.parse(test_code_str)
        
        # Find the check() function if it exists
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef) and node.name == "check":
                # Extract assertions from the function body
                for stmt in node.body:
                    if isinstance(stmt, ast.Assert):
                        # Convert AST assertion back to source code string
                        assert_code = ast.unparse(stmt)
                        
                        # Replace 'candidate' with actual function name if provided
                        if entry_point_name:
                            assert_code = assert_code.replace('candidate(', f'{entry_point_name}(')
                        
                        assertions.append(assert_code)
                break
        
        return assertions
    except SyntaxError:
        return []


def test_code(code, test_imports, tests, timeout=2):
    """
    Test generated code against test cases.
    Wrapper around safe_exec_with_tests with proper error handling.
    """
    return safe_exec_with_tests(code, test_imports, tests, timeout=timeout)


def get_tests_from_record(record):
    """Return (test_imports, tests_list) from a record safely for both MBPP and HumanEval."""
    
    # For MBPP: has test_imports and test_list
    if "test_list" in record and record.get("test_list"):
        test_imports = record.get("test_imports", []) or record.get("imports", []) or []
        tests_list = record.get("test_list", []) or record.get("tests", []) or []
    # For HumanEval: has test field with check() function
    elif "test" in record and record.get("test"):
        test_code_str = record.get("test", "")
        test_imports = [
            line.strip()
            for line in test_code_str.split("\n")
            if line.strip().startswith(("import ", "from "))
        ]
        # Extract entry_point name for HumanEval (used to replace 'candidate' in assertions)
        entry_point = record.get("entry_point", None)
        tests_list = extract_assertions_from_check_function(test_code_str, entry_point_name=entry_point)
    else:
        test_imports = []
        tests_list = []
    
    # Normalize to list of strings
    if isinstance(test_imports, str):
        test_imports = [test_imports]
    if isinstance(tests_list, str):
        tests_list = [tests_list]
    
    return test_imports, tests_list


### Watermark Detection

In [117]:
def detect_watermark(original_code, generated_code, green_letters, red_letters, gamma):
    """
    WATERMARK DETECTION

    - Extracts identifier tokens and comment tokens SEPARATELY
    - Merges with proper deduplication to prevent double-counting
    - Single unified p-exact score for decision and ROC ranking
    - Returns separate metrics for transparency but calculates ONE p-value

    Deduplication Process:
    - If 'total' is both an identifier AND in a comment, it counts as 2 tokens
    - But we track: id_token_count + comment_token_count - overlap_count
    - Green count follows the same deduplication logic

    Note: 
    - Dedent code before parsing to handle indented canonical solutions
    """
    
    orig_stats = get_starting_letters(original_code, gamma)

    gen_stats = get_starting_letters(generated_code, gamma)

    # ✅ UNIFIED DETECTION: Use p_exact for both decision and all returned stats
    def is_watermarked_unified(p_exact, threshold=P_THRESHOLD):
        """Simple threshold-based decision using p_exact."""
        if math.isnan(p_exact):
            return False
        return p_exact < threshold

    return {
        # Original code stats - matching exact original CSV column format
        "original_token_count": orig_stats.get("total_token_count", 0),
        "original_green_count": orig_stats.get("total_green_count", 0),
        "original_z_score": orig_stats.get("z_score", 0.0),
        "original_p_exact": orig_stats.get("p_exact", 1.0),
        "original_p_unified": orig_stats.get("p_exact", 1.0),  # Use same as p_exact
        "original_score": orig_stats.get("score", 0.0),
        "original_method_used": "binomial",
        "original_preferred_ratio": orig_stats.get("total_green_count", 0)
        / max(1, orig_stats.get("total_token_count", 1)),
        "original_is_watermarked": is_watermarked_unified(
            orig_stats.get("p_exact", 1.0)
        ),
        "original_unique_starts": "".join(
            sorted(set(orig_stats.get("id_starting_letters", [])))
        ),
        # Generated code stats - matching exact original CSV column format
        "generated_token_count": gen_stats.get("total_token_count", 0),
        "generated_green_count": gen_stats.get("total_green_count", 0),
        "generated_z_score": gen_stats.get("z_score", 0.0),
        "generated_p_exact": gen_stats.get("p_exact", 1.0),
        "generated_p_unified": gen_stats.get("p_exact", 1.0),  # Use same as p_exact
        "generated_score": gen_stats.get("score", 0.0),
        "generated_method_used": "binomial",
        "generated_preferred_ratio": gen_stats.get("total_green_count", 0)
        / max(1, gen_stats.get("total_token_count", 1)),
        "generated_is_watermarked": is_watermarked_unified(
            gen_stats.get("p_exact", 1.0)
        ),
        "generated_unique_starts": "".join(
            sorted(set(gen_stats.get("id_starting_letters", [])))
        ),
    }

In [118]:
import math
from scipy.stats import binomtest, norm, binom, chi2


def calculate_z_score(token_count, green_count, gamma):
    """Calculate z-score for green token count (standard normal approximation)."""
    
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(
        gamma * (1 - gamma) * token_count
    )


def calculate_gamma(
    letter_counts: Dict[str, int], total_count: int, green_letters: set
) -> float:
    """

    Formula:
        γ = Σ(frequency[l] / total_count) for l ∈ green_letters

    Args:
        letter_counts: Dictionary mapping letter -> count from COMBINED datasets
        total_count: Total identifiers from COMBINED datasets
        green_letters: Set of green letters (most frequent)

    Returns:
        float: Empirical proportion of green letters (0.0 to 1.0)
    """
    if total_count == 0:
        return 0.0

    gamma = 0.0

    for letter in green_letters:
        if letter in letter_counts:
            p_letter = letter_counts[letter] / total_count
            gamma += p_letter

    # return 0.39
    return gamma


def get_detection_score(token_count, green_count, gamma):
    """
    Use only exact binomial p-value (no z-test).

    Returns p-exact and unified score.
    - p_exact = binom.sf(green_count - 1, token_count, gamma)  (exact binomial test)
    - score = -log10(p_exact)  (for ROC analysis and ranking)
    """
    
    p_exact = binom.sf(green_count - 1, token_count, gamma)

    # Score for ROC: higher = more evidence of watermark
    score = -np.log10(np.clip(p_exact, 1e-300, 1.0))

    return {
        "p_exact": p_exact,
        "score": score,
        "token_count": token_count,
    }

### Green List Processing


In [119]:
import hashlib
import textwrap
import json
import random
from typing import Set, List, Tuple, Dict
from collections import defaultdict

# Module-level variable to track current green set size
CURRENT_GREEN_SET_SIZE = None


def calculate_character_frequencies(df: pd.DataFrame, code_column: str = 'code', top_n: int = 18) -> Tuple[Dict[str, int], int, Dict[str, list]]:
    """
    Process corpus of code samples and calculate letter frequencies.
    """
    letter_counts = {chr(ord('a') + i): 0 for i in range(26)}  # a-z
    identifiers_by_letter = {chr(ord('a') + i): [] for i in range(26)}  # Track identifiers for each letter
    total_count = 0
    
    valid_samples = 0
    skipped_samples = 0

    freq_sum = defaultdict(int)
    
    for idx, row in df.iterrows():
        try:
            code = row[code_column]
            code = textwrap.dedent(code)
            
            # Skip if code is empty or not a string
            if pd.isna(code) or not isinstance(code, str) or not code.strip():
                skipped_samples += 1
                continue
            
            # Extract starting letters and identifiers
            letters, identifiers = extract_identifier_starting_letters(code)
            
            if letters:
                valid_samples += 1
                total_count += len(letters)
                
                # Update frequency counts and track identifiers
                for ident in identifiers:
                    # Get first alphabetic letter
                    first_letter = None
                    for char in ident:
                        if char.isalpha():
                            first_letter = char.lower()
                            break
                    
                    if first_letter and first_letter in letter_counts:
                        letter_counts[first_letter] += 1
                        
                        identifiers_by_letter[first_letter].append(ident)
        
        except Exception as e:
            skipped_samples += 1
            continue

    for letter in sorted(letter_counts.keys()):
        cnt = letter_counts[letter]
        freq_sum[letter] += cnt

    sorted_chars = sorted(freq_sum.items(), key=lambda x: x[1], reverse=True)
    top_n = [char for char, _ in sorted_chars[:top_n]]

    # return letter_counts, total_count, identifiers_by_letter, top_n
    return top_n


def cryptographically_selected_greenlist(
    secret_key: str, candidate_letters: List[str], g_min: int = 8, g_max: int = 18
) -> Tuple[Set[str], int]:
    """
    Construct green-set from secret key and candidate letters.

    Derives green-set size from SHA256(key), then generates a deterministic
    Fisher-Yates permutation using SHA256(key || "green-set-selection")
    as seed. Returns the first |G| letters from the permutation.

    Args:
        secret_key: Secret key string
        candidate_letters: Candidate letters sorted by frequency (highest first)
        g_min: Minimum green-set size (default: 8)
        g_max: Maximum green-set size (default: 18)

    Returns:
        Tuple[Set[str], int]: Green-set and its size

    Raises:
        ValueError: If parameters are invalid
    """

    if len(candidate_letters) < g_max:
        raise ValueError(
            f"Candidate set size ({len(candidate_letters)}) must be >= g_max ({g_max})"
        )
    if g_min < 1 or g_max > len(candidate_letters) or g_min > g_max:
        raise ValueError(
            f"Invalid bounds: g_min={g_min}, g_max={g_max}, |C|={len(candidate_letters)}"
        )

    # Derive green-set size from first hash
    h1 = hashlib.sha256(secret_key.encode()).digest()
    u1_int = int.from_bytes(h1[0:4], byteorder="big")
    u1 = u1_int / (2**32)
    size_g = int(g_min + u1 * (g_max - g_min + 1))
    size_g = max(g_min, min(g_max, size_g))

    # Derive permutation seed from second hash
    h2_input = secret_key.encode() + b"green-set-selection-v1"
    h2 = hashlib.sha256(h2_input).digest()
    seed = int.from_bytes(h2[0:8], byteorder="big")

    # Generate Fisher-Yates permutation
    shuffled = candidate_letters.copy()
    random.Random(seed).shuffle(shuffled)

    return set(shuffled[:size_g]), size_g


def get_red_green_sets(
    secret_key: str = SEED_KEY, base_dir: str = BASE_DIR
) -> Tuple[set, set, int]:
    """
    Get red and green letter sets using frequency-based selection.
    """
    global CURRENT_GREEN_SET_SIZE

    humaneval_path = f"{base_dir}/results/dataset/humaneval_letter_freqs.json"
    mbpp_path = f"{base_dir}/results/dataset/mbpp_letter_freqs.json"

    # candidates = get_frequent_candidates(humaneval_path, mbpp_path, top_n=18)d
    dataset_df = pd.read_json(DATASET_PATH, lines=True)
    candidates = calculate_character_frequencies(dataset_df, top_n=18)

    green_set, size_g = cryptographically_selected_greenlist(secret_key, candidates, g_min=6, g_max=10)
    red_set = set(candidates) - green_set

    # Track current green set size globally
    CURRENT_GREEN_SET_SIZE = size_g

    return green_set, red_set, size_g


### Prompt Construction


In [120]:
SYSTEM_PROMPT = '''
## Additional Instruction:
#### Green Letter List: {green_words}
#### Red Letter List: {red_words}

### Command:
Generate code following the given instructions:
1. Green Letter Enriched Identifier: When generating identifiers (local variables, function parameters, private helper functions, internal class attributes, temporary variables) & comments, start the words with letters from the 'Green Letter List'.
2. Correct & Relevant: Generate correct code following the problem statement.
3. About comments: Add brief comments only to clarify complex logic, but do not over-comment or exaggerate. Avoid tokens that starts with the Red List letters.
4. About Method Name: Write the method name as mentioned in the given test case.
5. Warning: Do not mention or explain the renaming rules in your output.
6. Others: Provide explanation justifying why it will work outside the code block & at the end of the response. If generated code fails tests, then follow the explanation and write down the correct code. Never include the given test cases, explanation or assertions inside the codeblocks.
'''


In [121]:
PROBLEM_TEMPLATE = (
    "You are a helpful code assistant that can teach a junior developer how to code."
    "Your language of choice is Python. Generate the Python code for the following task enclosed in ```python```\n\n"
    "##Prompt:\n{prompt}\n\n"
    "##Test Cases:\n{tests}\n\n"
)

### Load LLM

In [122]:
from google import genai
from google.genai import types
import os
import re
from pathlib import Path
import time
import random
from google.api_core import exceptions as google_exceptions

from dotenv import load_dotenv

load_dotenv()

True

In [123]:
API_KEY = os.getenv('GEMINI_API_KEY')

if not API_KEY:
    raise RuntimeError(
        "Missing API key. Please set GEMINI_API_KEY or API_KEY in your environment (e.g., .env)."
    )

gemini_client = genai.Client(api_key=API_KEY)

print(gemini_client.models.count_tokens(
    model=MODEL_PATH,
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text="text")],
        )
    ]
))

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) total_tokens=2 cached_content_token_count=None


In [124]:
import boto3

claude_client = boto3.client(
    "bedrock-runtime",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name="us-east-1",
)

### Iterative Code Generation

In [125]:
def generate_response_by_gemini(system_prompt, problem_statement, max_retries=5):
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=problem_statement)],
        )
    ]

    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=MODEL_PATH,
                contents=contents,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt
                ),
            )

            input_tokens = response.usage_metadata.prompt_token_count
            output_tokens = response.usage_metadata.candidates_token_count
            total_tokens = response.usage_metadata.total_token_count

            return {
                "text": response.text.strip(),
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": total_tokens,
            }

        except Exception:
            raise

In [126]:
def generate_response_by_claude(
    prompt: str,
):
    """Generate response from Claude via AWS Bedrock with optional token tracking."""

    model_id = os.getenv("DEFAULT_MODEL", "us.anthropic.claude-sonnet-4-6")
    print("USING MODEL:", model_id)
    text = ""

    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2048,
        "messages": [{"role": "user", "content": prompt}],
    }

    response = claude_client.invoke_model(
        modelId=model_id, body=json.dumps(request_body), contentType="application/json"
    )

    response_body = json.loads(response["body"].read())
    
    if "content" in response_body and len(response_body["content"]) > 0:
        text = response_body["content"][0]["text"]

    usage = response_body.get("usage", {})
    return {
        "text": text,
        "input_tokens": usage.get("input_tokens", 0),
        "output_tokens": usage.get("output_tokens", 0),
        "total_tokens": usage.get("input_tokens", 0)
        + usage.get("output_tokens", 0),
    }

In [128]:
def generate_code(record, feedback_prompt=""):
    
    global green_letters, red_letters

    task_id = record["task_id"]
    problem_query = record["prompt"]
    testcases = "\n".join(record['test_list']) if DATASET == 'mbpp' else []
    full_prompt = PROBLEM_TEMPLATE.format(prompt=problem_query, tests=testcases)

    #? CHOOSE SYSTEM PROMPT BASED ON WATERMARKING FLAG
    if APPLY_WATERMARKING:
        green_letters, red_letters, _ = get_red_green_sets(
            secret_key=SEED_KEY, base_dir=BASE_DIR
        )
        system_instruction = SYSTEM_PROMPT.format(
            green_words=green_letters, red_words=red_letters
        )
    else:
        # Use generic prompt without watermarking constraints
        system_instruction = ""

    if len(feedback_prompt) > 0:
        full_prompt = full_prompt + "\n\n" + feedback_prompt

    full_prompt_with_system = f"{system_instruction}\n\n{full_prompt}"

    #? GENERATE RESPONSE (MAKE SURE TO CHANGE THE "MODEL_NAME" ENV VAR TO SWITCH MODELS)
    # result = generate_response_by_gemini(
    #     system_instruction, full_prompt
    # )

    result = generate_response_by_claude(
        full_prompt_with_system
    )

    #? EXTRACT CODE AND EXPLANATION FROM RESPONSE
    full_text = result["text"].strip()
    input_tokens = result["input_tokens"]
    output_tokens = result["output_tokens"]

    code_blocks = re.findall(r"```python(.*?)```", full_text, re.DOTALL)
    actual_code_blocks = [block.strip() for block in code_blocks if block.strip()]
    code_text = actual_code_blocks[-1] if actual_code_blocks else ""
    explanation_text = re.sub(
        r"```python.*?```", "", full_text, flags=re.DOTALL
    ).strip()

    return {
        "code": code_text,
        "explanation": explanation_text,
        "full_response": full_text,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
    }

In [ ]:
def evaluate_candidate(record, generated_code):
    """
    Evaluate generated code for correctness and watermark presence, returning a comprehensive result dictionary.
    """
    
    global CURRENT_GREEN_SET_SIZE

    task_id = record["task_id"]
    
    #? DETERMINE DATASET TYPE
    is_mbpp = "test_list" in record
    is_humaneval = "test" in record
    
    #? CHECK CORRECTNESS IF ENABLED
    if CHECK_CORRECTNESS:
        # Get tests from record (handles both MBPP and HumanEval)
        test_imports, tests = get_tests_from_record(record)
        
        if is_mbpp and tests:
            generated_code = update_method_name(generated_code, tests)
        
        test_result = test_code(generated_code, test_imports, tests, timeout=2)
        passed, failed = test_result.get("result", (0, 0))
    else:
        passed, failed = 0, 0
        test_result = {"error": "Correctness check disabled"}

    is_correct = (failed == 0) if CHECK_CORRECTNESS else None

    total_tests = passed + failed if CHECK_CORRECTNESS else 0
    pass_rate = (
        (passed / total_tests * 100.0)
        if (CHECK_CORRECTNESS and total_tests > 0)
        else 0.0
    )
    all_passed = (failed == 0) if CHECK_CORRECTNESS else None

    eval_res = {
        "task_id": task_id,
        "tests_passed": passed,
        "tests_failed": failed,
        "total_tests": total_tests,
        "pass_rate": pass_rate,
        "all_passed": all_passed if CHECK_CORRECTNESS else False,
        "correctness": is_correct if CHECK_CORRECTNESS else None,
        "error_message": test_result.get("error", ""),
    }

    #? DETERMINE IF WE NEED TO CALCULATE WATERMARK METRICS
    should_calculate_watermark = CHECK_CORRECTNESS or APPLY_WATERMARKING

    if should_calculate_watermark:
        
        original_code = record.get("canonical_solution", "") or record.get("code", "")
        green_letters, red_letters, size_g = get_red_green_sets(
            secret_key=SEED_KEY, base_dir=BASE_DIR
        )
        
        CURRENT_GREEN_SET_SIZE = size_g

        letter_freqs, total_identifiers = load_character_frequency_list(green_letters, BASE_DIR)
        GAMMA = calculate_gamma(letter_freqs, total_identifiers, green_letters)
        # GAMMA = 0.88
        print(f"⚠️ Calculated GAMMA: {GAMMA:.4f} based on character frequency list")

        #! Full watermark detection
        detection_result = detect_watermark(
            original_code, generated_code, green_letters, red_letters, GAMMA
        )

        # Add all watermark stats to results
        eval_res.update(detection_result)

        # Only set meets_z if APPLY_WATERMARKING is True (this affects the stopping condition)
        # But we always calculate the metrics
        eval_res["meets_z"] = (
            bool(detection_result.get("generated_is_watermarked", False))
            if APPLY_WATERMARKING
            else False
        )
    else:
        # Neither correctness checking nor watermarking - skip all watermark detection
        eval_res.update(
            {
                "original_z_score": float("nan"),
                "original_p_exact": float("nan"),
                "original_p_unified": float("nan"),
                "original_score": float("nan"),
                "original_method_used": "none",
                "original_preferred_ratio": float("nan"),
                "original_token_count": 0,
                "original_green_count": 0,
                "original_is_watermarked": False,
                "meets_z": False,
                "original_unique_starts": "",
                "generated_z_score": float("nan"),
                "generated_p_exact": float("nan"),
                "generated_p_unified": float("nan"),
                "generated_score": float("nan"),
                "generated_method_used": "none",
                "generated_preferred_ratio": float("nan"),
                "generated_token_count": 0,
                "generated_green_count": 0,
                "generated_is_watermarked": False,
                "generated_unique_starts": "",
            }
        )

    return eval_res

In [129]:
from datetime import datetime

def execute_pipeline(
    record, max_iterations=ITER_CAP, verbose=False, iterative_mode=ITERATIVE_MODE
):
    """
    Generate code following the configured behavior:

    Args:
        record: Task record with prompt and test information
        max_iterations: Maximum number of iterations to attempt (ignored if iterative_mode=False)
        verbose: Whether to print verbose output
        iterative_mode: If True, use iterative generation with feedback; if False, generate once

    Returns: (selected_result_dict, token_tracking_dict)
    """

    best = None
    selected = None
    token_tracking = {"iterations": []}
    feedback_prompt = ""  # Accumulate feedback across iterations
    green_letters, red_letters, _ = (
        get_red_green_sets(secret_key=SEED_KEY, base_dir=BASE_DIR)
        if APPLY_WATERMARKING
        else (set(), set(), 0)
    )

    # In one-shot mode, only generate once regardless of conditions
    iterations_to_run = 1 if not iterative_mode else max_iterations

    for it in range(iterations_to_run):
        print(f"\n{'='*60}\n[Iteration {it+1}/{iterations_to_run}]\n{'='*60}")

        # --- Generate code ---
        gen = generate_code(record, feedback_prompt=feedback_prompt)
        code = gen["code"]
        reasoning_text = gen.get("explanation", "")

        iter_tokens = {
            "iteration": it,
            "input_tokens": gen.get("input_tokens", 0),
            "output_tokens": gen.get("output_tokens", 0),
            "total_tokens": gen.get("total_tokens", 0),
        }
        token_tracking["iterations"].append(iter_tokens)

        eval_res = evaluate_candidate(record, code)
        eval_res.update(
            {
                "iteration": it,
                "reasoning_text": reasoning_text,
                "full_llm_response": gen["full_response"],
                "input_tokens": iter_tokens["input_tokens"],
                "output_tokens": iter_tokens["output_tokens"],
            }
        )

        # Determine stopping condition based on control constants
        if iterative_mode:
            # Iterative mode: check stopping conditions
            if CHECK_CORRECTNESS and APPLY_WATERMARKING:
                # Both correctness and watermarking required
                eval_res["stopping_condition_met"] = (
                    eval_res["correctness"] and eval_res["meets_z"]
                )
            elif CHECK_CORRECTNESS and not APPLY_WATERMARKING:
                # Only correctness required (no watermarking)
                eval_res["stopping_condition_met"] = eval_res["correctness"]
            elif not CHECK_CORRECTNESS and APPLY_WATERMARKING:
                # Only watermarking required (no correctness check)
                eval_res["stopping_condition_met"] = eval_res["meets_z"]
            else:
                # No checking required (CHECK_CORRECTNESS=False, APPLY_WATERMARKING=False)
                # Just generate once
                eval_res["stopping_condition_met"] = True
        else:
            # One-shot mode: always stop after first generation
            eval_res["stopping_condition_met"] = True

        eval_res["code"] = code

        # ✅ DEBUG: Print evaluation metrics
        print(f"\n📊 EVALUATION METRICS:")
        if CHECK_CORRECTNESS:
            print(f"   Correctness: {eval_res['correctness']}")
            print(
                f"   Tests Passed: {eval_res['tests_passed']}/{eval_res['tests_passed'] + eval_res['tests_failed']}"
            )
        if APPLY_WATERMARKING:
            print(f"   Generated Token Count: {eval_res['generated_token_count']}")
            print(f"   Generated Green Count: {eval_res['generated_green_count']}")
            print(f"   P-Value (p_exact): {eval_res['generated_p_exact']:.10f}")
            print(f"   Score (-log10(p)): {eval_res['generated_score']:.4f}")
            print(f"   Meets Watermark (p < {P_THRESHOLD:.6f}): {eval_res['meets_z']}")
        print(f"   Stopping Condition Met: {eval_res['stopping_condition_met']}\n")

        # --- Prepare feedback if not stopping ---
        feedback_prompt = ""

        # Early stopping
        if eval_res["stopping_condition_met"]:
            selected = eval_res
            token_tracking["total_input_tokens"] = sum(
                t["input_tokens"] for t in token_tracking["iterations"]
            )
            token_tracking["total_output_tokens"] = sum(
                t["output_tokens"] for t in token_tracking["iterations"]
            )
            token_tracking["total_tokens"] = (
                token_tracking["total_input_tokens"]
                + token_tracking["total_output_tokens"]
            )
            if verbose:
                msg = f"[{record['task_id']}] ✅ Stop: "
                if CHECK_CORRECTNESS:
                    msg += f"Correct ({eval_res['tests_passed']}/{eval_res['total_tests']})"
                if APPLY_WATERMARKING:
                    if CHECK_CORRECTNESS:
                        msg += " & "
                    p_val = eval_res.get("generated_p_exact", float("nan"))
                    msg += f"p_exact={p_val:.6f} < {P_THRESHOLD}"
                print(msg)
            break

        # Only prepare feedback in iterative mode
        if iterative_mode:
            # Prepare feedback based on failures
            if CHECK_CORRECTNESS and not eval_res["correctness"]:
                error_log = eval_res.get("error_message", "Unknown error")
                feedback_prompt = f"❌ The previous code execution failed.\nAnalyze the error step by step:\n1. Review the error log: \n{error_log}\n2. Consider the previous reasoning: {reasoning_text}\n\nNow write the explanation why it failed. \nThen generate the corrected version of the code that fixes the issues. \nEnsure the code is executable following the problem requirements."
                print(f"\n🔄 FEEDBACK REASON: Code correctness failed")
            elif APPLY_WATERMARKING and not eval_res["meets_z"]:
                # Watermark fidelity failed - code is correct but doesn't have watermark
                p_exact = eval_res.get("generated_p_exact", 1.0)
                token_count = eval_res.get("generated_token_count", 0)
                green_count = eval_res.get("generated_green_count", 0)
                score = eval_res.get("generated_score", float("nan"))

                feedback_prompt = f"Your provided code is correct BUT FAILED to create WATERMARK (p-value={p_exact:.6f}, P_THRESHOLD={P_THRESHOLD:.6f}).\nGreen Token Count={green_count}/{token_count}. Only REFACTOR the code to use more green-letter identifiers ({', '.join(list(green_letters))}) in identifiers and comments.\n"
                print(f"\n🔄 FEEDBACK REASON: Watermark fidelity failed")

            if feedback_prompt:
                print(f"\n💬 FEEDBACK PROMPT:\n{feedback_prompt}\n")

        # Update best result (using unified p-value score for ranking: higher score = stronger watermark)
        if APPLY_WATERMARKING:
            current_score = eval_res.get("generated_score", -1e9)
            if math.isnan(current_score):
                current_score = -1e9
            best_score = best.get("generated_score", -1e9) if best else -1e9
            if math.isnan(best_score):
                best_score = -1e9

            if best is None or (
                (
                    eval_res.get("correctness", False)
                    and not best.get("correctness", False)
                )
                if CHECK_CORRECTNESS
                else True or (current_score > best_score)
            ):
                best = eval_res
        else:
            # If not applying watermarking, track best by correctness
            if CHECK_CORRECTNESS:
                if best is None or (
                    eval_res.get("correctness", False)
                    and not best.get("correctness", False)
                ):
                    best = eval_res
            else:
                best = eval_res

    if selected is None:
        selected = best
        token_tracking["total_input_tokens"] = sum(
            t["input_tokens"] for t in token_tracking["iterations"]
        )
        token_tracking["total_output_tokens"] = sum(
            t["output_tokens"] for t in token_tracking["iterations"]
        )
        token_tracking["total_tokens"] = (
            token_tracking["total_input_tokens"] + token_tracking["total_output_tokens"]
        )

    return (selected or {}, token_tracking)

### RUN


In [130]:
import time


def process_dataset(df, output_dir):
    Path(output_dir).parent.mkdir(exist_ok=True)
    results = []
    all_token_tracking = []
    total_exp_input_tokens = 0
    total_exp_output_tokens = 0

    for idx, record in df.iterrows():
        task_id = record.get("task_id") or record.get("id")
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        try:
            sel, token_info = execute_pipeline(
                record, max_iterations=ITER_CAP, verbose=True, iterative_mode=ITERATIVE_MODE
            )
            code = sel.get("code", "") if sel else ""
            iteration_used = sel.get("iteration") if sel and "iteration" in sel else None

            # Save code
            output_file = out_dir / f"{task_id}.py"

                
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(code or "")
            
            # Track tokens
            total_exp_input_tokens += token_info.get("total_input_tokens", 0)
            total_exp_output_tokens += token_info.get("total_output_tokens", 0)
            
            # Store results
            result_row = {
                "task_id": task_id,
                "iteration_used": iteration_used,
                **sel
            }
            results.append(result_row)
            all_token_tracking.append({
                "task_id": task_id,
                **token_info
            })
            
        except Exception as e:
            print(f"❌ Error processing {task_id}: {e}")
            import traceback
            traceback.print_exc()
    
    # Save results to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"\n✅ Results saved to {RESULTS_CSV}")
    print(f"📊 Total tokens used: {total_exp_input_tokens + total_exp_output_tokens}")
    
    return results_df

In [131]:
results = process_dataset(df, OUTPUT_DIR)


[Iteration 1/5]
USING MODEL: us.anthropic.claude-sonnet-4-6
⚠️ Calculated GAMMA: 0.5552 based on character frequency list

📊 EVALUATION METRICS:
   Correctness: True
   Tests Passed: 10/10
   Generated Token Count: 8
   Generated Green Count: 8
   P-Value (p_exact): 0.0090225332
   Score (-log10(p)): 2.0447
   Meets Watermark (p < 0.024998): True
   Stopping Condition Met: True

[39] ✅ Stop: Correct (10/10) & p_exact=0.009023 < 0.024997895148220435

[Iteration 1/5]
USING MODEL: us.anthropic.claude-sonnet-4-6
⚠️ Calculated GAMMA: 0.5552 based on character frequency list

📊 EVALUATION METRICS:
   Correctness: True
   Tests Passed: 4/4
   Generated Token Count: 4
   Generated Green Count: 3
   P-Value (p_exact): 0.3994360741
   Score (-log10(p)): 0.3986
   Meets Watermark (p < 0.024998): False
   Stopping Condition Met: False


🔄 FEEDBACK REASON: Watermark fidelity failed

💬 FEEDBACK PROMPT:
Your provided code is correct BUT FAILED to create WATERMARK (p-value=0.399436, P_THRESHOLD=0.024